## ToDo

- 前提：original_data内の.csvは、それぞれノードIDを共有する複数のグラフを表している
- タスク1：まずfb_friends.csvの中身を元にノードのモジュール検出を行い、適当な数のモジュールに分けてラベル付けしたい（このネットワークは時間変化していない）
- タスク2：そのうえでbt_symmetric.csv, calls.csv, sms.csvはそれぞれ最初の時点からの経過分数で時間変化しているが、分数ではなく日数に直して記述したい（1日は1440分）
- タスク3：以下の形式のdata/copenhagen/copenhagen.txtおよびdata/copenhagen/node2label.txtを作成したい：
　- data/copenhagen/copenhagen.txt：<source> <target> <timestamp>
　- data/copenhagen/node2label.txt：<node> <label>


In [ ]:
import pandas as pd
import networkx as nx
import os

# プロジェクトルートを取得（ULSEディレクトリ）
# ノートブックはcopenhagen/ディレクトリにあるので、親ディレクトリがプロジェクトルート
current_dir = os.getcwd()
if os.path.basename(current_dir) == "copenhagen":
    # ノートブックがcopenhagen/ディレクトリで実行されている場合
    project_root = os.path.dirname(current_dir)
    data_dir = os.path.join(current_dir, "original_data")
else:
    # プロジェクトルートで実行されている場合
    project_root = current_dir
    data_dir = os.path.join(project_root, "copenhagen", "original_data")

# 出力ディレクトリの作成（プロジェクトルートからの相対パス）
output_dir = os.path.join(project_root, "data", "copenhagen")
os.makedirs(output_dir, exist_ok=True)

print(f"プロジェクトルート: {project_root}")
print(f"データディレクトリ: {data_dir}")
print(f"出力ディレクトリ: {output_dir}")


In [ ]:
# タスク1: fb_friends.csvからグラフを作成し、モジュール検出を行ってラベル付け
print("タスク1: fb_friends.csvからモジュール検出を実行中...")

# fb_friends.csvを読み込み（ヘッダー行が#で始まっているため、namesで明示的に指定）
fb_friends_path = os.path.join(data_dir, "fb_friends.csv")
df_fb = pd.read_csv(fb_friends_path, comment='#', names=['user_a', 'user_b'])

# グラフを作成
G = nx.Graph()
for _, row in df_fb.iterrows():
    G.add_edge(row['user_a'], row['user_b'])

print(f"ノード数: {G.number_of_nodes()}")
print(f"エッジ数: {G.number_of_edges()}")

# コミュニティ検出
from networkx.algorithms import community
communities_generator = community.greedy_modularity_communities(G)
communities = {}
for i, comm in enumerate(communities_generator):
    for node in comm:
        communities[node] = i
print(f"検出されたコミュニティ数: {len(communities_generator)}")

# ノードとラベルのマッピングを作成
node_to_label = communities
print(f"ラベル付けされたノード数: {len(node_to_label)}")


タスク1: fb_friends.csvからモジュール検出を実行中...
ノード数: 800
エッジ数: 6429
python-louvainが見つかりません。networkxのgreedy_modularity_communitiesを使用します。
検出されたコミュニティ数: 7
ラベル付けされたノード数: 800


In [ ]:
# タスク2: 時間データを分数から日数に変換（1日=1440分）
print("\nタスク2: 時間データを分数から日数に変換中...")

def convert_minutes_to_days(minutes):
    """分数を日数に変換（1日=1440分）"""
    return int(float(minutes) / 1440.0)

# 各CSVファイルを読み込み、時間を変換
all_edges = []

# bt_symmetric.csv
print("bt_symmetric.csvを処理中...")
bt_path = os.path.join(data_dir, "bt_symmetric.csv")
df_bt = pd.read_csv(bt_path, comment='#', names=['timestamp', 'user_a', 'user_b', 'rssi'])
# user_aとuser_bが-1や-2の場合はスキップ
df_bt = df_bt[(df_bt['user_a'] >= 0) & (df_bt['user_b'] >= 0)]
df_bt['timestamp_days'] = df_bt['timestamp'].apply(convert_minutes_to_days)
for _, row in df_bt.iterrows():
    all_edges.append((int(row['user_a']), int(row['user_b']), row['timestamp_days']))
print(f"  {len(df_bt)} エッジを追加")

# calls.csv
print("calls.csvを処理中...")
calls_path = os.path.join(data_dir, "calls.csv")
df_calls = pd.read_csv(calls_path)
df_calls['timestamp_days'] = df_calls['timestamp'].apply(convert_minutes_to_days)
for _, row in df_calls.iterrows():
    all_edges.append((int(row['caller']), int(row['callee']), row['timestamp_days']))
print(f"  {len(df_calls)} エッジを追加")

# sms.csv
print("sms.csvを処理中...")
sms_path = os.path.join(data_dir, "sms.csv")
df_sms = pd.read_csv(sms_path)
df_sms['timestamp_days'] = df_sms['timestamp'].apply(convert_minutes_to_days)
for _, row in df_sms.iterrows():
    all_edges.append((int(row['sender']), int(row['recipient']), row['timestamp_days']))
print(f"  {len(df_sms)} エッジを追加")

print(f"\n合計エッジ数: {len(all_edges)}")

# エッジを時系列でソート
all_edges.sort(key=lambda x: x[2])  # timestampでソート
print("エッジを時系列でソートしました")



タスク2: 時間データを分数から日数に変換中...
bt_symmetric.csvを処理中...
  2426279 エッジを追加
calls.csvを処理中...
  3600 エッジを追加
sms.csvを処理中...
  24333 エッジを追加

合計エッジ数: 2454212
エッジを時系列でソートしました


In [8]:
# タスク3: copenhagen.txtとnode2label.txtを作成
print("\nタスク3: 出力ファイルを作成中...")

# copenhagen.txtを作成（<source> <target> <timestamp>の形式）
copenhagen_txt_path = os.path.join(output_dir, "copenhagen.txt")
with open(copenhagen_txt_path, 'w') as f:
    for source, target, timestamp in all_edges:
        f.write(f"{source} {target} {timestamp}\n")
print(f"copenhagen.txtを作成しました: {len(all_edges)} エッジ")

# node2label.txtを作成（<node> <label>の形式）
# fb_friendsから検出したコミュニティラベルを使用
# 時系列データに含まれるがfb_friendsに含まれないノードは、デフォルトラベル0を割り当て
node2label_txt_path = os.path.join(output_dir, "node2label.txt")

# すべてのノードを収集
all_nodes = set()
for source, target, _ in all_edges:
    all_nodes.add(source)
    all_nodes.add(target)

# ラベル付け
with open(node2label_txt_path, 'w') as f:
    for node in sorted(all_nodes):
        label = node_to_label.get(node, 0)  # fb_friendsにないノードはラベル0
        f.write(f"{node} {label}\n")

print(f"node2label.txtを作成しました: {len(all_nodes)} ノード")
print(f"  ラベル数: {len(set(node_to_label.values()))}")
print(f"\n完了！出力先: {output_dir}")



タスク3: 出力ファイルを作成中...
copenhagen.txtを作成しました: 2454212 エッジ
node2label.txtを作成しました: 742 ノード
  ラベル数: 7

完了！出力先: data/copenhagen
